## Descarga e importaciones de librerias

In [1]:
!pip install pymupdf sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 20.2 MB/s eta 0:00:00


In [12]:
import fitz, json, os, re
from google.colab import drive
from sentence_transformers import SentenceTransformer

drive.mount("/content/drive")

DRIVE_BASE  = "/content/drive/MyDrive/rag_rioja"
PDF_FOLDER  = f"{DRIVE_BASE}/pdfs"
JSON_FOLDER = f"{DRIVE_BASE}/jsons"

os.makedirs(PDF_FOLDER,  exist_ok=True)
os.makedirs(JSON_FOLDER, exist_ok=True)

print(f"📄 PDFs disponibles: {os.listdir(PDF_FOLDER)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📄 PDFs disponibles: ['boletin_25.pdf', 'fito_15.pdf']


## Limpieza (del pdf en español)

In [13]:
def limpiar_texto(texto):
    """Limpia el texto extraído del PDF."""
    # Elimina saltos de línea dentro de párrafos
    texto = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto)
    # Colapsa espacios múltiples
    texto = re.sub(r' +', ' ', texto)
    # Elimina cabeceras/pies típicos de boletines (números de página solos)
    texto = re.sub(r'\n\s*\d+\s*\n', '\n', texto)
    return texto.strip()

def extraer_bloques_por_pagina(pdf_path):
    """Extrae texto página a página conservando metadatos."""
    doc     = fitz.open(pdf_path)
    bloques = []
    for num_pagina, pagina in enumerate(doc, start=1):
        texto = pagina.get_text("text")
        texto = limpiar_texto(texto)
        if texto:
            bloques.append({
                "texto"  : texto,
                "pagina" : num_pagina
            })
    return bloques

## Chunkgin (particionado del texto )

In [14]:
def chunking_semantico(bloques, chunk_size=600, overlap=80):
    """
    Divide los bloques en chunks respetando párrafos.
    Prioriza no cortar a mitad de párrafo.
    """
    chunks = []
    buffer = ""
    pagina_actual = 1

    for bloque in bloques:
        pagina_actual = bloque["pagina"]
        parrafos = [p.strip() for p in bloque["texto"].split("\n\n") if p.strip()]

        for parrafo in parrafos:
            # Si el párrafo solo ya supera el tamaño, lo divide por frases
            if len(parrafo) > chunk_size:
                frases = re.split(r'(?<=[.!?])\s+', parrafo)
                for frase in frases:
                    if len(buffer) + len(frase) <= chunk_size:
                        buffer += " " + frase
                    else:
                        if buffer.strip():
                            chunks.append({
                                "texto"  : buffer.strip(),
                                "pagina" : pagina_actual
                            })
                        # Overlap: conserva el final del buffer anterior
                        buffer = buffer[-overlap:] + " " + frase if overlap else frase
            else:
                if len(buffer) + len(parrafo) <= chunk_size:
                    buffer += "\n\n" + parrafo
                else:
                    if buffer.strip():
                        chunks.append({
                            "texto"  : buffer.strip(),
                            "pagina" : pagina_actual
                        })
                    buffer = buffer[-overlap:] + "\n\n" + parrafo if overlap else parrafo

    # Último buffer
    if buffer.strip():
        chunks.append({
            "texto"  : buffer.strip(),
            "pagina" : pagina_actual
        })

    return chunks


## Código  principal

In [15]:
def procesar_pdf(pdf_path):
    """Pipeline completo: PDF → chunks → JSON."""
    nombre_pdf = os.path.basename(pdf_path)
    nombre_json = nombre_pdf.replace(".pdf", "_chunks.json")
    output_path = f"{JSON_FOLDER}/{nombre_json}"

    # Saltar si ya existe
    if os.path.exists(output_path):
        print(f"⏭️  {nombre_pdf} ya procesado, saltando")
        return

    print(f"⚙️  Procesando {nombre_pdf}...")
    bloques = extraer_bloques_por_pagina(pdf_path)
    chunks  = chunking_semantico(bloques)

    # Añadir metadatos a cada chunk
    for i, chunk in enumerate(chunks):
        chunk["fuente"]          = nombre_pdf
        chunk["chunk_id"]        = i
        chunk["total_caracteres"] = len(chunk["texto"])

    resultado = {
        "archivo_origen" : nombre_pdf,
        "total_chunks"   : len(chunks),
        "chunks"         : chunks
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)

    print(f"  ✅ {len(chunks)} chunks → {output_path}")
    return resultado

## Main (Ejecución de todo el código de limpieza)

In [16]:
pdfs = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]
jsons_existentes = [f for f in os.listdir(JSON_FOLDER) if f.endswith(".json")]

print(f"📄 PDFs encontrados:    {len(pdfs)}")
print(f"📦 JSONs ya generados:  {len(jsons_existentes)}")
print()

for nombre_pdf in pdfs:
    ruta = f"{PDF_FOLDER}/{nombre_pdf}"
    procesar_pdf(ruta)

print(f"\n✅ JSONs disponibles en Drive: {os.listdir(JSON_FOLDER)}")

📄 PDFs encontrados:    2
📦 JSONs ya generados:  0

⚙️  Procesando boletin_25.pdf...
  ✅ 12 chunks → /content/drive/MyDrive/rag_rioja/jsons/boletin_25_chunks.json
⚙️  Procesando fito_15.pdf...
  ✅ 15 chunks → /content/drive/MyDrive/rag_rioja/jsons/fito_15_chunks.json

✅ JSONs disponibles en Drive: ['boletin_25_chunks.json', 'fito_15_chunks.json']


## Revisión del Output .Json (Opcional)

In [17]:
json_test = os.listdir(JSON_FOLDER)[0]
with open(f"{JSON_FOLDER}/{json_test}", "r") as f:
    datos = json.load(f)

print(f"📄 Archivo: {datos['archivo_origen']}")
print(f"📦 Total chunks: {datos['total_chunks']}")
print()
for i, chunk in enumerate(datos["chunks"][:3]):   # muestra los 3 primeros
    print(f"── CHUNK {i+1} (pág. {chunk['pagina']}, {chunk['total_caracteres']} chars) ──")
    print(chunk["texto"][:300])
    print()

📄 Archivo: boletin_25.pdf
📦 Total chunks: 12

── CHUNK 1 (pág. 1, 505 chars) ──
Sección de Protección de Cultivos  Finca La Grajera (Edificio administrativo)  T. 941 29 14 55  boletin.avisos@larioja.org  www.larioja.org/agricultura Boletín de avisos fitosanitarios Nº 25  11 de diciembre de 2025 Herbicidas en cereales de invierno II Los cultivos cerealistas comienzan una nu

── CHUNK 2 (pág. 1, 590 chars) ──
aves y, al mismo tiempo, se continúa con las últimas siembras de trigo y cebada. En las parcelas más desarrolladas que tuvieron una sementera temprana y donde, no se han realizado un tratamiento herbicida en preemergencia o en postemergencia temprana del cultivo sembrado (boletín nº 23 de 2025), se 

── CHUNK 3 (pág. 1, 504 chars) ──
o, se observan campos de cereal donde el tratamiento herbicida resulta ineficaz. Hay que diferenciar si es debido por una mala ejecución (derivas, días de viento, humedad insuficiente, etc.), una inadecuada decisión técnica (mezclas no compatibles,